In [2]:
!pip install requests pandas

import requests
import json
import pandas as pd
from pathlib import Path

API_URL = "https://jsonplaceholder.typicode.com/users"
RAW_DATA_PATH = "data/raw_data.json"
PROCESSED_DATA_PATH = "data/clean_data.csv"
Path("data").mkdir(exist_ok=True)

print("Environment ready ✅")
print(f"API URL: {API_URL}")
print(f"RAW DATA PATH: {RAW_DATA_PATH}")

Defaulting to user installation because normal site-packages is not writeable
Environment ready ✅
API URL: https://jsonplaceholder.typicode.com/users
RAW DATA PATH: data/raw_data.json


In [3]:
def extract_data():
    print("Extracting data from API...")

    response = requests.get(API_URL)
    response.raise_for_status()

    data = response.json()

    with open(RAW_DATA_PATH, "w") as f:
        json.dump(data, f, indent=4)

    print(f"Saved {len(data)} records to {RAW_DATA_PATH}")


    return data

In [5]:
import pandas as pd
import json
import os

# Create data folder
os.makedirs("data", exist_ok=True)

print("Folder created successfully ✅")

def transform_data(data):
    print("Transforming data...")

    # ---------------------------
    # LOAD RAW DATA
    # ---------------------------
    df = pd.json_normalize(data)

    # ---------------------------
    # CLEAN COLUMN NAMES
    # ---------------------------
    df.columns = (
        df.columns
        .str.lower()
        .str.replace(".", "_", regex=False)
        .str.strip()
    )

    # ---------------------------
    # SELECT IMPORTANT COLUMNS
    # ---------------------------
    df = df[
        [
            "id",
            "name",
            "username",
            "email",
            "address_city",
            "company_name"
        ]
    ]

    # ---------------------------
    # DATA TYPE ENFORCEMENT
    # ---------------------------
    df["id"] = df["id"].astype(int)
    df["name"] = df["name"].astype(str)
    df["email"] = df["email"].astype(str)

    # ---------------------------
    # TEXT STANDARDIZATION
    # ---------------------------
    df["name"] = df["name"].str.title()
    df["username"] = df["username"].str.lower()
    df["email"] = df["email"].str.lower()
    df["address_city"] = df["address_city"].str.title()
    df["company_name"] = df["company_name"].str.title()

    # ---------------------------
    # HANDLE MISSING VALUES
    # ---------------------------
    df.fillna(
        {
            "address_city": "Unknown",
            "company_name": "Unknown"
        },
        inplace=True
    )

    # ---------------------------
    # REMOVE DUPLICATES
    # ---------------------------
    df.drop_duplicates(subset=["id"], inplace=True)

    # ---------------------------
    # FEATURE ENGINEERING
    # ---------------------------

    # Extract email domain
    df["email_domain"] = df["email"].str.split("@").str[-1]

    # Name length feature
    df["name_length"] = df["name"].str.len()

    # Username length
    df["username_length"] = df["username"].str.len()

    # City + Company combined field
    df["location_company"] = (
        df["address_city"] + " - " + df["company_name"]
    )

    # ---------------------------
    # DATA QUALITY CHECKS
    # ---------------------------

    # Remove invalid emails
    df = df[df["email"].str.contains("@", na=False)]

    # Remove empty names
    df = df[df["name"].str.strip() != ""]

    # ---------------------------
    # SORT DATASET
    # ---------------------------
    df.sort_values(by="id", inplace=True)

    # Reset index
    df.reset_index(drop=True, inplace=True)

    # ---------------------------
    # SAVE CLEAN DATA
    # ---------------------------
    df.to_csv(PROCESSED_DATA_PATH, index=False)

    print("Transformation completed successfully ✅")
    print(f"Cleaned data to {PROCESSED_DATA_PATH}")
    


    return df

Folder created successfully ✅


In [7]:
import pandas as pd
from sqlalchemy import create_engine 
import urllib

def load_to_sql(df):
    print("Loading data into SQL Server...")

    # 1. Setup paths and connection string
    PROCESSED_DATA_PATH = "data/clean_data.csv"
    
    # Use double backslashes for the server name instance
    conn_str = (
        "DRIVER={ODBC Driver 17 for SQL Server};"
        "SERVER=DESKTOP-I111GFR\\SQLEXPRESS;"
        "DATABASE=ETL_DB;"
        "Trusted_Connection=yes;"
        "TrustServerCertificate=yes;"
    )
    
    # 2. Create SQLAlchemy Engine (Crucial: requires URL encoding for the string)
    quoted_conn = urllib.parse.quote_plus(conn_str)
    engine = create_engine(f"mssql+pyodbc:///?odbc_connect={quoted_conn}")

    try:
        # 3. Load the cleaned CSV and Upload to SQL
        # 'append' adds data to existing table; 'replace' recreates it

        df.to_sql(
            name="Users", 
            con=engine, 
            if_exists="replace", 
            index=False,
            chunksize=1000
        )
        
        print("Load completed successfully.")
        

    except Exception as e:
        print(f"An error occurred: {e}")

In [8]:
def run_pipeline():
    print("Starting ETL pipeline...")

    try:
        # Step 1: Extract
        print("Extracting data...")
        raw_data = extract_data()

        # Step 2: Transform
        print("Transforming data...")
        transformed_df = transform_data(raw_data)

        # Step 3: Load
        print("Loading data into SQL Server...")
        load_to_sql(transformed_df)

        print("ETL pipeline completed successfully.")
        

    except Exception as e:
        print(f"Pipeline failed: {e}")


if __name__ == "__main__":
    run_pipeline()


Starting ETL pipeline...
Extracting data...
Extracting data from API...
Saved 10 records to data/raw_data.json
Transforming data...
Transforming data...
Transformation completed successfully ✅
Cleaned data to data/clean_data.csv
Loading data into SQL Server...
Loading data into SQL Server...
Load completed successfully.
ETL pipeline completed successfully.
